# Thực nghiệm phân rã hiệu năng GPU và tối ưu hóa Mixed Precision trên CatVTON (Tesla T4)

Notebook thực hiện nghiên cứu thực nghiệm độc lập trên môi trường Kaggle GPU (`NVIDIA Tesla T4`, kiến trúc Turing, Compute Capability `7.5`):
1. **Đo lường phân rã hiệu năng GPU**: Thời gian khởi động nguội (`cold-start`), tiền xử lý (`AutoMasker`), suy diễn khuếch tán 20 bước (`DPM++ 2M Karras`), hậu xử lý giải mã `VAE` và đỉnh bộ nhớ đồ họa (`Peak VRAM`).
2. **Nghiên cứu đối sánh Mixed Precision**: Khảo sát thực nghiệm trực tiếp giữa `FP32`, `FP16` và `BF16` trên tập ảnh người mẫu thực tế của `CatVTON`, tính toán độ tương đồng cấu trúc (`SSIM`) và tỷ số tín hiệu trên nhiễu đỉnh (`PSNR`) so với mốc đối chứng `FP32`.
3. **Xuất gói minh chứng thực nghiệm**: Tự động tạo lưới ảnh đối sánh 4 khung hình (`[Người mẫu | Trang phục | FP32 | FP16]`), xuất bảng số liệu `precision_ablation.csv` và đóng gói tệp nén `/kaggle/working/vton_research_results.zip`.

In [ ]:
# Cell 1: Thiet lap moi truong va clone ma nguon CatVTON
import os

os.chdir("/kaggle/working")
if not os.path.isdir("CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
%cd /kaggle/working/CatVTON

# Loai bo cac goi ghim phien ban xung dot voi PyTorch/CUDA cai san tren Kaggle
!grep -viE '^(gradio|huggingface_hub|torch|torchvision|accelerate|diffusers|peft)' requirements.txt > requirements_research.txt
!pip install -q -r requirements_research.txt
!pip install -q "diffusers>=0.30.0" "accelerate>=0.33.0" "transformers>=4.44.0" "peft>=0.13.0" "huggingface_hub>=0.34.0,<2.0" fvcore iopath yacs pycocotools omegaconf cloudpickle av scikit-image
print("[OK] Thiet lap moi truong hoan tat.")

In [ ]:
# Cell 2: Nap checkpoint va khoi tao AutoMasker tren CUDA
import glob
import io
import json
import os
import shutil
import time
import zipfile
import numpy as np
import torch
from PIL import Image
from diffusers import DPMSolverMultistepScheduler
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

torch.backends.cudnn.benchmark = True


def find_local_dir(root, folder_name):
    if not os.path.isdir(root):
        return None
    for match in glob.glob(os.path.join(root, "**", folder_name), recursive=True):
        if os.path.isdir(match) and os.listdir(match):
            return match
    return None


DATASET_ROOT = "/kaggle/input/catvton-weights"
WORKING_ROOT = "/kaggle/working/catvton_weights"

catvton_local = find_local_dir(DATASET_ROOT, "CatVTON") or os.path.join(WORKING_ROOT, "CatVTON")
if os.path.isdir(catvton_local) and os.listdir(catvton_local):
    repo_path = catvton_local
else:
    repo_path = snapshot_download(repo_id="zhengchong/CatVTON", local_dir=catvton_local)

sd_local = find_local_dir(DATASET_ROOT, "sd-inpainting") or os.path.join(WORKING_ROOT, "sd-inpainting")
if os.path.isdir(sd_local) and os.listdir(sd_local):
    base_ckpt_path = sd_local
else:
    base_ckpt_path = snapshot_download(
        repo_id="booksforcharlie/stable-diffusion-inpainting",
        local_dir=sd_local,
    )

print(f"[CKPT] CatVTON: {repo_path}")
print(f"[CKPT] SD-Inpainting: {base_ckpt_path}")
mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device="cuda",
)
print("[OK] AutoMasker khoi tao thanh cong tren CUDA.")

In [ ]:
# Cell 3: Nghien cuu doi sanh Mixed Precision (FP32 vs FP16 vs BF16) bang anh thuc te
import gc
import glob
import os
import shutil
import time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

PRECISION_MODES = ["fp32", "fp16", "bf16"]
DTYPE_MAPPING = {
    "fp32": "no",   # CatVTON utils.py anh xa "no" -> torch.float32
    "fp16": "fp16",
    "bf16": "bf16",
}

OUTPUT_RESEARCH_DIR = "/kaggle/working/research_results"
shutil.rmtree(OUTPUT_RESEARCH_DIR, ignore_errors=True)
os.makedirs(OUTPUT_RESEARCH_DIR, exist_ok=True)

# Nap tap anh nguoi mau va trang phuc thuc te trong thu muc demo cua CatVTON
repo_root = "/kaggle/working/CatVTON"
person_demo_dir = os.path.join(repo_root, "resource/demo/example/person/women")
cloth_demo_dir = os.path.join(repo_root, "resource/demo/example/condition/upper")

person_files = sorted(glob.glob(os.path.join(person_demo_dir, "*.*")))
cloth_files = sorted(glob.glob(os.path.join(cloth_demo_dir, "*.*")))

SAMPLE_PAIRS = []
if person_files and cloth_files:
    SAMPLE_PAIRS.append({
        "pair_id": "P-01",
        "category": "upper",
        "person_path": person_files[0],
        "cloth_path": cloth_files[0],
    })
    if len(person_files) > 1 and len(cloth_files) > 1:
        SAMPLE_PAIRS.append({
            "pair_id": "P-02",
            "category": "upper",
            "person_path": person_files[1],
            "cloth_path": cloth_files[1],
        })
    print(f"[DATA] Tim thay {len(SAMPLE_PAIRS)} cap anh nguoi mau va trang phuc thuc te trong CatVTON demo.")
else:
    raise FileNotFoundError("Khong tim thay anh demo thuc te trong /kaggle/working/CatVTON/resource/demo/example.")

precision_results = []
fp32_reference_images = {}
generated_images_map = {}

for p_mode in PRECISION_MODES:
    print(f"\n=================== THUC NGHIEM PRECISION: {p_mode.upper()} ===================")
    p_line = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    t_load_start = time.perf_counter()
    try:
        dtype_key = DTYPE_MAPPING[p_mode]
        p_line = CatVTONPipeline(
            base_ckpt=base_ckpt_path,
            attn_ckpt=repo_path,
            attn_ckpt_version="mix",
            weight_dtype=init_weight_dtype(dtype_key),
            use_tf32=True if p_mode == "fp32" else False,
            device="cuda",
            skip_safety_check=True,
        )
        p_line.noise_scheduler = DPMSolverMultistepScheduler.from_config(
            p_line.noise_scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++",
        )
        torch.cuda.synchronize()
        cold_start_s = round(time.perf_counter() - t_load_start, 3)
        print(f"[{p_mode.upper()}] Cold-start nap pipeline: {cold_start_s}s")

        pre_times, inf_times, post_times, total_times = [], [], [], []

        for sp in SAMPLE_PAIRS:
            pid = sp["pair_id"]
            p_img = Image.open(sp["person_path"]).convert("RGB")
            c_img = Image.open(sp["cloth_path"]).convert("RGB")

            # 1. Preprocess
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            p_prep = resize_and_crop(p_img, (768, 1024))
            c_prep = resize_and_padding(c_img, (768, 1024))
            mask = automasker(p_prep, sp["category"])["mask"]
            mask = mask_processor.blur(mask, blur_factor=9)
            torch.cuda.synchronize()
            t_pre = time.perf_counter() - t0

            # 2. Inference (co dinh seed=42 de doi sanh cong bang giua cac che do precision)
            generator = torch.Generator(device="cuda").manual_seed(42)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            with torch.inference_mode():
                res_img = p_line(
                    image=p_prep,
                    condition_image=c_prep,
                    mask=mask,
                    num_inference_steps=20,
                    guidance_scale=4.5,
                    generator=generator,
                )[0]
            torch.cuda.synchronize()
            t_inf = time.perf_counter() - t1

            # 3. Postprocess
            t2 = time.perf_counter()
            out_p = os.path.join(OUTPUT_RESEARCH_DIR, f"{pid}_{p_mode}.png")
            res_img.save(out_p)
            t_post = time.perf_counter() - t2

            pre_times.append(t_pre)
            inf_times.append(t_inf)
            post_times.append(t_post)
            total_times.append(t_pre + t_inf + t_post)

            generated_images_map[f"{pid}_{p_mode}"] = res_img
            if p_mode == "fp32":
                fp32_reference_images[pid] = np.array(res_img)

        peak_vram = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)
        mean_inf = round(float(np.mean(inf_times)), 2)
        mean_pre = round(float(np.mean(pre_times)), 2)
        mean_post = round(float(np.mean(post_times)), 2)
        mean_tot = round(float(np.mean(total_times)), 2)

        # Doi sanh SSIM / PSNR voi moc doi chung FP32
        ssim_scores, psnr_scores = [], []
        for sp in SAMPLE_PAIRS:
            pid = sp["pair_id"]
            if pid in fp32_reference_images:
                curr_arr = np.array(Image.open(os.path.join(OUTPUT_RESEARCH_DIR, f"{pid}_{p_mode}.png")))
                ref_arr = fp32_reference_images[pid]
                score_s = ssim(curr_arr, ref_arr, channel_axis=2, data_range=255)
                score_p = (
                    float("inf")
                    if np.array_equal(curr_arr, ref_arr)
                    else psnr(curr_arr, ref_arr, data_range=255)
                )
                ssim_scores.append(score_s)
                psnr_scores.append(score_p)

        mean_ssim = round(float(np.mean(ssim_scores)), 4) if ssim_scores else (1.0 if p_mode == "fp32" else "N/A")
        if p_mode == "fp32":
            mean_psnr = "inf"
        else:
            finite_psnr = [v for v in psnr_scores if np.isfinite(v)]
            mean_psnr = round(float(np.mean(finite_psnr)), 2) if finite_psnr else "N/A"

        note = (
            "Baseline reference"
            if p_mode == "fp32"
            else ("Native Turing Tensor Cores" if p_mode == "fp16" else "Emulated / Unsupported on T4")
        )
        precision_results.append({
            "precision_mode": p_mode.upper(),
            "torch_dtype": f"torch.{p_mode}",
            "cold_start_s": cold_start_s,
            "mean_preprocess_s": mean_pre,
            "mean_inference_s": mean_inf,
            "mean_postprocess_s": mean_post,
            "mean_total_s": mean_tot,
            "peak_vram_gb": peak_vram,
            "mean_ssim_vs_fp32": mean_ssim,
            "mean_psnr_vs_fp32": mean_psnr,
            "status_note": note,
        })
        print(
            f"[{p_mode.upper()}] Mean Inference: {mean_inf}s | "
            f"Peak VRAM: {peak_vram} GB | SSIM vs FP32: {mean_ssim} | PSNR: {mean_psnr}"
        )
    except Exception as exc:
        err_msg = str(exc)
        print(f"[{p_mode.upper()} ERROR / INCOMPATIBLE]: {err_msg}")
        precision_results.append({
            "precision_mode": p_mode.upper(),
            "torch_dtype": f"torch.{p_mode}",
            "cold_start_s": "N/A",
            "mean_preprocess_s": "N/A",
            "mean_inference_s": "N/A",
            "mean_postprocess_s": "N/A",
            "mean_total_s": "N/A",
            "peak_vram_gb": "N/A",
            "mean_ssim_vs_fp32": "N/A",
            "mean_psnr_vs_fp32": "N/A",
            "status_note": f"Hardware Incompatibility on T4: {err_msg[:60]}",
        })
    finally:
        if p_line is not None:
            del p_line
        gc.collect()
        torch.cuda.empty_cache()

# Tao anh luoi doi sanh 4 khung hinh [Nguoi mau | Trang phuc | FP32 | FP16]
try:
    for sp in SAMPLE_PAIRS:
        pid = sp["pair_id"]
        fp32_key = f"{pid}_fp32"
        fp16_key = f"{pid}_fp16"
        if fp32_key in generated_images_map and fp16_key in generated_images_map:
            p_im = resize_and_crop(Image.open(sp["person_path"]).convert("RGB"), (768, 1024))
            c_im = resize_and_padding(Image.open(sp["cloth_path"]).convert("RGB"), (768, 1024))
            r_32 = generated_images_map[fp32_key].resize((768, 1024))
            r_16 = generated_images_map[fp16_key].resize((768, 1024))

            grid = Image.new("RGB", (768 * 4, 1024))
            grid.paste(p_im, (0, 0))
            grid.paste(c_im, (768, 0))
            grid.paste(r_32, (768 * 2, 0))
            grid.paste(r_16, (768 * 3, 0))

            grid_out_path = os.path.join(OUTPUT_RESEARCH_DIR, f"{pid}_comparison_grid.png")
            grid.save(grid_out_path)
            print(f"[VISUAL GRID] Da tao anh doi sanh 4 khung hinh: {grid_out_path}")
except Exception as exc:
    print(f"[WARN] Khong the tao visual grid: {exc}")

df_precision = pd.DataFrame(precision_results)
df_precision.to_csv(os.path.join(OUTPUT_RESEARCH_DIR, "precision_ablation.csv"), index=False)
print("\n[HOAN TAT] Da xuat precision_ablation.csv:")
print(df_precision.to_string())

In [ ]:
# Cell 4: Dong goi toan bo ket qua nghien cuu ra tep zip de tai ve
import os
import zipfile

OUTPUT_RESEARCH_DIR = "/kaggle/working/research_results"
zip_path = "/kaggle/working/vton_research_results.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_RESEARCH_DIR):
        for file_name in sorted(files):
            full_p = os.path.join(root, file_name)
            arc_p = os.path.relpath(full_p, OUTPUT_RESEARCH_DIR)
            zipf.write(full_p, arc_p)

size_mb = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
print(f"[XUAT THANH CONG] Tep ket qua nghien cuu san sang tai: {zip_path}")
print(f"Kich thuoc tep: {size_mb} MB")
print("Danh sach tep duoc dong goi:")
for f_name in sorted(os.listdir(OUTPUT_RESEARCH_DIR)):
    print(" -", f_name)